# Notebook 5 — Solutions: Connectome-Constrained Neural Network

**BINFX410 · Chapter 10 · Connectomics**

Reference solutions for the five exercises in `05_connectome_neural_network.ipynb`.  
Run notebooks 1–4 first to generate `../data/connectome_architecture.pkl`.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import Adam

import networkx as nx

DATA_DIR = Path('../data')
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(DATA_DIR / 'connectome_architecture.pkl', 'rb') as f:
    arch = pickle.load(f)

A      = arch['adjacency_matrix']   # (N, N)
G      = arch['G']
N      = arch['n_neurons']
layers = arch['layer_groups']

print(f'Connectome: {N} neurons, {G.number_of_edges()} connections, device={DEVICE}')

In [ ]:
# ── Shared: dataset generation and training loop ──────────────────────────────

N_CLASSES = 4

def make_neural_activity_dataset(n_samples, n_neurons, n_classes, adjacency, seed=0):
    rng = np.random.default_rng(seed)
    class_patterns = (rng.uniform(0, 1, (n_classes, n_neurons)) > 0.6).astype(float)
    P_local = adjacency / (adjacency.sum(axis=1, keepdims=True) + 1e-8)
    propagated = np.zeros_like(class_patterns)
    for c in range(n_classes):
        act = class_patterns[c].copy()
        for _ in range(3):
            act = 0.5 * act + 0.5 * (act @ P_local)
        propagated[c] = act
    X_list, y_list = [], []
    per_class = n_samples // n_classes
    for c in range(n_classes):
        for _ in range(per_class):
            X_list.append(propagated[c] + rng.normal(0, 0.15, n_neurons))
            y_list.append(c)
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)
    idx = rng.permutation(len(X))
    return torch.from_numpy(X[idx]), torch.from_numpy(y[idx])


X_train, y_train = make_neural_activity_dataset(800, N, N_CLASSES, A, seed=0)
X_val,   y_val   = make_neural_activity_dataset(200, N, N_CLASSES, A, seed=99)
train_ld = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_ld   = DataLoader(TensorDataset(X_val,   y_val),   batch_size=32, shuffle=False)


def train_model(model, train_loader, val_loader, n_epochs=60, lr=1e-3, label=''):
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    history   = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    t0 = time.time()
    for epoch in range(1, n_epochs + 1):
        model.train()
        tl = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            optimizer.step()
            tl += F.cross_entropy(model(xb), yb).item()
        model.eval()
        vl = correct = total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                vl     += F.cross_entropy(logits, yb).item()
                correct += (logits.argmax(1) == yb).sum().item()
                total   += yb.size(0)
        history['train_loss'].append(tl/len(train_loader))
        history['val_loss'].append(vl/len(val_loader))
        history['val_acc'].append(correct/total)
        if epoch % 20 == 0 or epoch == 1:
            print(f'[{label}] Ep{epoch:3d}: val_acc={history["val_acc"][-1]:.3f}')
    print(f'  Total time: {time.time()-t0:.1f}s')
    return history


print('Shared utilities ready.')

---
## Exercise 5.1 — Truly Sparse ConnectomeLinear

**Task:** Modify `ConnectomeLinear` so masked weights are **not registered as parameters** at all — only active weights exist in memory.

In [ ]:
class ConnectomeLinear(nn.Module):
    """Original ConnectomeLinear — all weights exist but masked in forward."""
    def __init__(self, adjacency_matrix, bias=True):
        super().__init__()
        n_in  = adjacency_matrix.shape[0]
        n_out = adjacency_matrix.shape[1]
        mask  = torch.tensor((adjacency_matrix > 0).T.astype(np.float32))
        self.register_buffer('mask', mask)
        self.weight = nn.Parameter(torch.empty(n_out, n_in))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        self.bias_param = nn.Parameter(torch.zeros(n_out)) if bias else None

    def forward(self, x):
        return F.linear(x, self.weight * self.mask, self.bias_param)

    @property
    def n_active_weights(self): return int(self.mask.sum().item())


class SparseBiologicalLinear(nn.Module):
    """
    Truly sparse connectome-constrained linear layer.

    Only active (biological) weights are registered as parameters.
    Uses torch.sparse or index-based scatter to avoid storing dead weights.

    Implementation strategy:
      - Store active weight values as a 1-D nn.Parameter (n_active_weights,)
      - Store the (row, col) indices of active connections as buffers
      - In forward(), scatter the parameter values into a full weight matrix
        (or use torch.sparse_coo_tensor for a memory-efficient matmul)
    """

    def __init__(self, adjacency_matrix: np.ndarray, bias: bool = True):
        super().__init__()
        n_in  = adjacency_matrix.shape[0]
        n_out = adjacency_matrix.shape[1]
        self.n_in  = n_in
        self.n_out = n_out

        # Find active connections: mask[j, i] = 1 if A[i,j] > 0
        mask_np = (adjacency_matrix > 0).T  # (n_out, n_in)
        rows, cols = np.where(mask_np)       # active (j, i) pairs

        self.register_buffer('row_idx', torch.tensor(rows, dtype=torch.long))
        self.register_buffer('col_idx', torch.tensor(cols, dtype=torch.long))
        self.n_active = len(rows)

        # Only active weights as parameters — no ghost parameters
        active_weights = torch.empty(self.n_active)
        nn.init.kaiming_uniform_(active_weights.unsqueeze(0), a=np.sqrt(5))
        self.active_weights = nn.Parameter(active_weights.squeeze())

        self.bias_param = nn.Parameter(torch.zeros(n_out)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Reconstruct the weight matrix from active values
        W = torch.zeros(self.n_out, self.n_in, device=x.device, dtype=x.dtype)
        W[self.row_idx, self.col_idx] = self.active_weights
        return F.linear(x, W, self.bias_param)

    @property
    def n_params(self):
        return self.n_active + (self.n_out if self.bias_param is not None else 0)


# ── Parameter count comparison ────────────────────────────────────────────────
original_layer = ConnectomeLinear(A)
sparse_layer   = SparseBiologicalLinear(A)

orig_params = sum(p.numel() for p in original_layer.parameters() if p.requires_grad)
sparse_params = sum(p.numel() for p in sparse_layer.parameters() if p.requires_grad)

print('Parameter comparison (single layer):')
print(f'  Original ConnectomeLinear : {orig_params:,} parameters ({N}×{N} weight + {N} bias)')
print(f'  SparseBiologicalLinear    : {sparse_params:,} parameters ({sparse_layer.n_active} active weights + {N} bias)')
print(f'  Memory saved              : {orig_params - sparse_params:,} ghost parameters eliminated')
print(f'  Sparsity                  : {1 - sparse_layer.n_active/(N*N):.1%}')

# ── Functional equivalence test ───────────────────────────────────────────────
test_input = torch.randn(4, N)
# Copy weights from sparse to original for comparison
with torch.no_grad():
    W_sparse = torch.zeros(N, N)
    W_sparse[sparse_layer.row_idx, sparse_layer.col_idx] = sparse_layer.active_weights
    original_layer.weight.data[:] = W_sparse
    if original_layer.bias_param is not None and sparse_layer.bias_param is not None:
        original_layer.bias_param.data[:] = sparse_layer.bias_param.data

out_orig   = original_layer(test_input)
out_sparse = sparse_layer(test_input)
print(f'\nOutputs match: {torch.allclose(out_orig, out_sparse, atol=1e-5)}')
print(f'Max difference: {(out_orig - out_sparse).abs().max().item():.2e}')

In [ ]:
# ── Build networks using the sparse layer and compare training ─────────────────

class SparseConnectomeNet(nn.Module):
    def __init__(self, adjacency, n_classes):
        super().__init__()
        self.layer1 = SparseBiologicalLinear(adjacency, bias=True)
        self.layer2 = SparseBiologicalLinear(adjacency, bias=True)
        self.head   = nn.Linear(adjacency.shape[0], n_classes)
        self.drop   = nn.Dropout(0.3)
    def forward(self, x):
        x = self.drop(F.relu(self.layer1(x)))
        x = self.drop(F.relu(self.layer2(x)))
        return self.head(x)


class DenseConnectomeNet(nn.Module):
    """Original (masked) ConnectomeNet for comparison."""
    def __init__(self, adjacency, n_classes):
        super().__init__()
        self.layer1 = ConnectomeLinear(adjacency, bias=True)
        self.layer2 = ConnectomeLinear(adjacency, bias=True)
        self.head   = nn.Linear(adjacency.shape[0], n_classes)
        self.drop   = nn.Dropout(0.3)
    def forward(self, x):
        x = self.drop(F.relu(self.layer1(x)))
        x = self.drop(F.relu(self.layer2(x)))
        return self.head(x)


print('Training Dense (masked) ConnectomeNet...')
dense_net  = DenseConnectomeNet(A, N_CLASSES).to(DEVICE)
hist_dense = train_model(dense_net,  train_ld, val_ld, n_epochs=40, label='Dense')

print('\nTraining Sparse (no ghost params) ConnectomeNet...')
sparse_net  = SparseConnectomeNet(A, N_CLASSES).to(DEVICE)
hist_sparse = train_model(sparse_net, train_ld, val_ld, n_epochs=40, label='Sparse')

dense_params  = sum(p.numel() for p in dense_net.parameters() if p.requires_grad)
sparse_params2 = sum(p.numel() for p in sparse_net.parameters() if p.requires_grad)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_dense['val_acc'],  label=f'Dense (masked, {dense_params} params)', color='steelblue')
ax.plot(hist_sparse['val_acc'], label=f'Sparse (true, {sparse_params2} params)', color='tomato', linestyle='--')
ax.axhline(1/N_CLASSES, color='gray', linestyle=':', label='Random baseline')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy')
ax.set_title('Exercise 5.1 — Dense (Masked) vs. Truly Sparse ConnectomeNet')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Final accuracy — Dense : {hist_dense["val_acc"][-1]:.3f}')
print(f'Final accuracy — Sparse: {hist_sparse["val_acc"][-1]:.3f}')
print()
print('Note: accuracy should be similar (same effective computation).')
print('Benefit of true sparsity: fewer parameters stored in optimizer state')
print('(Adam stores m and v for each parameter), reducing memory footprint.')

---
## Exercise 5.2 — Biologically Meaningful Task: Sensory → Motor

**Task:** Design a task where sensory neurons (low in-degree) provide input and we predict which motor neurons (high out-degree) activate.

In [ ]:
# ── Identify sensory and motor neurons from the connectome ────────────────────
in_degrees  = dict(G.in_degree())
out_degrees = dict(G.out_degree())

# Sensory: low in-degree (receive few inputs from within the circuit)
# Motor  : high out-degree (send many outputs to the circuit)
# Interneuron: neither extreme

in_deg_vals  = np.array([in_degrees.get(n, 0)  for n in sorted(G.nodes())])
out_deg_vals = np.array([out_degrees.get(n, 0) for n in sorted(G.nodes())])

sensory_threshold = in_deg_vals.mean() - 0.5 * in_deg_vals.std()
motor_threshold   = out_deg_vals.mean() + 0.5 * out_deg_vals.std()

sensory_nodes = [n for n in G.nodes() if in_degrees.get(n, 0)  <= sensory_threshold]
motor_nodes   = [n for n in G.nodes() if out_degrees.get(n, 0) >= motor_threshold]

# Ensure we have both
if not sensory_nodes: sensory_nodes = [min(in_degrees, key=in_degrees.get)]
if not motor_nodes:   motor_nodes   = [max(out_degrees, key=out_degrees.get)]

print(f'Sensory neurons (low in-degree ≤ {sensory_threshold:.1f}): {sorted(sensory_nodes)}')
print(f'Motor neurons   (high out-degree ≥ {motor_threshold:.1f}): {sorted(motor_nodes)}')
print()

# ── Generate sensory → motor dataset ─────────────────────────────────────────
# Task: classify which subset of motor neurons is active based on sensory input pattern

def make_sensory_motor_dataset(n_samples, adjacency, sensory, motor, n_classes=4, seed=0):
    """
    Generate a dataset where:
    - Input:  activity patterns restricted to sensory neurons
    - Output: which of n_classes motor activation patterns results

    Activity propagates through the full connectome; the motor neuron
    activation at steady state is the classification target.
    """
    rng = np.random.default_rng(seed)
    n   = adjacency.shape[0]
    P_local = adjacency / (adjacency.sum(axis=1, keepdims=True) + 1e-8)

    # Define n_classes sensory input patterns (sparse, only sensory neurons active)
    class_inputs = np.zeros((n_classes, n), dtype=np.float32)
    for c in range(n_classes):
        n_active = max(1, len(sensory) // 2)
        active   = rng.choice(sensory, size=n_active, replace=False)
        class_inputs[c, active] = rng.uniform(0.5, 1.0, n_active)

    # Propagate through connectome (5 steps)
    class_steady = []
    for c in range(n_classes):
        act = class_inputs[c].copy()
        for _ in range(5):
            act = 0.3 * act + 0.7 * (act @ P_local)
        class_steady.append(act)

    X_list, y_list = [], []
    per_class = n_samples // n_classes
    for c in range(n_classes):
        for _ in range(per_class):
            # Input: sensory pattern with noise
            x = class_inputs[c].copy() + rng.normal(0, 0.1, n)
            # Zero out non-sensory neurons in input (only sensory neurons are observed)
            mask_sens = np.zeros(n); mask_sens[sensory] = 1
            x = x * mask_sens
            X_list.append(x.astype(np.float32))
            y_list.append(c)

    X = np.array(X_list); y = np.array(y_list, dtype=np.int64)
    idx = rng.permutation(len(X))
    return torch.from_numpy(X[idx]), torch.from_numpy(y[idx]), class_steady


N_CLASSES_SM = min(4, len(sensory_nodes) + 1)
X_sm_train, y_sm_train, steady = make_sensory_motor_dataset(
    800, A, sensory_nodes, motor_nodes, n_classes=N_CLASSES_SM, seed=0)
X_sm_val, y_sm_val, _ = make_sensory_motor_dataset(
    200, A, sensory_nodes, motor_nodes, n_classes=N_CLASSES_SM, seed=99)

sm_train_ld = DataLoader(TensorDataset(X_sm_train, y_sm_train), batch_size=32, shuffle=True)
sm_val_ld   = DataLoader(TensorDataset(X_sm_val,   y_sm_val),   batch_size=32, shuffle=False)

print(f'Sensory→Motor dataset: {X_sm_train.shape[0]} train, {X_sm_val.shape[0]} val')
print(f'Input dim: {X_sm_train.shape[1]} (only sensory neurons active)')

# Train a ConnectomeNet on this biologically grounded task
class ConnectomeNetSM(nn.Module):
    def __init__(self, adjacency, n_classes):
        super().__init__()
        n = adjacency.shape[0]
        mask = torch.tensor((adjacency > 0).T.astype(np.float32))
        self.register_buffer('mask', mask)
        self.weight1 = nn.Parameter(torch.empty(n, n)); nn.init.kaiming_uniform_(self.weight1)
        self.weight2 = nn.Parameter(torch.empty(n, n)); nn.init.kaiming_uniform_(self.weight2)
        self.head    = nn.Linear(n, n_classes)
    def forward(self, x):
        x = F.relu(F.linear(x, self.weight1 * self.mask))
        x = F.relu(F.linear(x, self.weight2 * self.mask))
        return self.head(x)

model_sm = ConnectomeNetSM(A, N_CLASSES_SM).to(DEVICE)
print('\nTraining ConnectomeNet on sensory→motor task...')
hist_sm = train_model(model_sm, sm_train_ld, sm_val_ld, n_epochs=60, label='Sensory→Motor')
print(f'Final accuracy: {hist_sm["val_acc"][-1]:.3f}  (random baseline: {1/N_CLASSES_SM:.3f}')

---
## Exercise 5.3 — Hebbian Plasticity

**Task:** Add `Δw = η_hebb * pre * post` to the gradient update. Does it improve convergence?

In [ ]:
class HebbianConnectomeLinear(nn.Module):
    """
    ConnectomeLinear with a Hebbian plasticity term.

    After each forward pass, stores pre- and post-synaptic activations.
    Call hebbian_update(eta) after optimizer.step() to apply the Hebbian rule:
        Δw[j,i] += eta * pre_activation[i] * post_activation[j]
    Only active (unmasked) weights are modified.
    """

    def __init__(self, adjacency_matrix, bias=True):
        super().__init__()
        n_in  = adjacency_matrix.shape[0]
        n_out = adjacency_matrix.shape[1]
        mask  = torch.tensor((adjacency_matrix > 0).T.astype(np.float32))
        self.register_buffer('mask', mask)
        self.weight = nn.Parameter(torch.empty(n_out, n_in))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        self.bias_param = nn.Parameter(torch.zeros(n_out)) if bias else None
        self._pre  = None   # stored during forward
        self._post = None

    def forward(self, x):
        self._pre  = x.detach()                                              # (B, n_in)
        out        = F.linear(x, self.weight * self.mask, self.bias_param)
        self._post = F.relu(out).detach()                                    # (B, n_out)
        return out

    def hebbian_update(self, eta=0.01):
        """Apply batch-averaged Hebbian weight increment."""
        if self._pre is None or self._post is None:
            return
        with torch.no_grad():
            # Outer product averaged over batch: (n_out, n_in)
            dW = torch.einsum('bi,bj->ij', self._post, self._pre) / self._pre.shape[0]
            # Only update active weights
            self.weight.data += eta * dW * self.mask


class HebbianConnectomeNet(nn.Module):
    def __init__(self, adjacency, n_classes):
        super().__init__()
        n = adjacency.shape[0]
        self.layer1 = HebbianConnectomeLinear(adjacency, bias=True)
        self.layer2 = HebbianConnectomeLinear(adjacency, bias=True)
        self.head   = nn.Linear(n, n_classes)
        self.drop   = nn.Dropout(0.3)

    def forward(self, x):
        x = self.drop(F.relu(self.layer1(x)))
        x = self.drop(F.relu(self.layer2(x)))
        return self.head(x)

    def hebbian_step(self, eta=0.005):
        self.layer1.hebbian_update(eta)
        self.layer2.hebbian_update(eta)


def train_with_hebbian(model, train_loader, val_loader, n_epochs=60, lr=1e-3, eta_hebb=0.005, label='Hebbian'):
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    history   = {'val_acc': [], 'train_loss': []}
    for epoch in range(1, n_epochs + 1):
        model.train()
        tl = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            optimizer.step()
            # Apply Hebbian update AFTER gradient step
            if eta_hebb > 0:
                model.hebbian_step(eta=eta_hebb)
            tl += loss.item()
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb).argmax(1)
                correct += (preds == yb).sum().item()
                total   += yb.size(0)
        history['train_loss'].append(tl/len(train_loader))
        history['val_acc'].append(correct/total)
        if epoch % 20 == 0 or epoch == 1:
            print(f'[{label}] Ep{epoch:3d}: val_acc={history["val_acc"][-1]:.3f}')
    return history


print('Training without Hebbian (η=0)...')
model_no_hebb = HebbianConnectomeNet(A, N_CLASSES).to(DEVICE)
hist_no_hebb  = train_with_hebbian(model_no_hebb, train_ld, val_ld, n_epochs=40,
                                   eta_hebb=0.0, label='No Hebbian')

print('\nTraining with Hebbian (η=0.005)...')
model_hebb = HebbianConnectomeNet(A, N_CLASSES).to(DEVICE)
hist_hebb  = train_with_hebbian(model_hebb, train_ld, val_ld, n_epochs=40,
                                eta_hebb=0.005, label='With Hebbian')

print('\nTraining with Hebbian (η=0.05)...')
model_hebb_strong = HebbianConnectomeNet(A, N_CLASSES).to(DEVICE)
hist_hebb_strong  = train_with_hebbian(model_hebb_strong, train_ld, val_ld, n_epochs=40,
                                       eta_hebb=0.05, label='Strong Hebbian')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_no_hebb['val_acc'],     label='η_hebb=0',     color='steelblue')
ax.plot(hist_hebb['val_acc'],        label='η_hebb=0.005', color='tomato')
ax.plot(hist_hebb_strong['val_acc'], label='η_hebb=0.05',  color='green')
ax.axhline(1/N_CLASSES, color='gray', linestyle=':', label='Random baseline')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation Accuracy')
ax.set_title('Exercise 5.3 — Effect of Hebbian Plasticity on Convergence')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nDiscussion:')
print('  Hebbian rule (Δw ∝ pre × post) strengthens connections between co-active neurons.')
print('  Small η: can provide a small boost to early convergence by reinforcing')
print('    features that reliably co-activate for the same class.')
print('  Large η: typically HURTS accuracy — Hebbian updates are unsupervised and')
print('    fight against the supervised gradient signal, destabilising training.')
print('  Best practice: use Hebbian as a regularisation term, not a primary learning rule.')

---
## Exercise 5.4 — Random Sparse Graph Ablation

**Task:** Run 5 different random sparse graphs at the same density as the connectome. Where does the real connectome fall?

In [ ]:
actual_density = (A > 0).sum() / (N * N)
print(f'Connectome density: {actual_density:.4f}')

N_RANDOM_SEEDS = 5
N_EPOCHS_ABL   = 40

# Train on the real connectome
class ConnectomeNetAbl(nn.Module):
    def __init__(self, adj, n_classes):
        super().__init__()
        n = adj.shape[0]
        mask = torch.tensor((adj > 0).T.astype(np.float32))
        self.register_buffer('mask', mask)
        self.w1 = nn.Parameter(torch.empty(n, n)); nn.init.kaiming_uniform_(self.w1)
        self.w2 = nn.Parameter(torch.empty(n, n)); nn.init.kaiming_uniform_(self.w2)
        self.head = nn.Linear(n, n_classes)
    def forward(self, x):
        x = F.relu(F.linear(x, self.w1 * self.mask))
        x = F.relu(F.linear(x, self.w2 * self.mask))
        return self.head(x)


print('Training on real connectome...')
real_net  = ConnectomeNetAbl(A, N_CLASSES).to(DEVICE)
hist_real = train_model(real_net, train_ld, val_ld, n_epochs=N_EPOCHS_ABL, label='Real connectome')
real_final_acc = hist_real['val_acc'][-1]

# Train on 5 random sparse graphs at the same density
random_final_accs = []
random_histories  = []

for seed in range(N_RANDOM_SEEDS):
    rng_np = np.random.default_rng(seed + 100)
    A_rand = (rng_np.uniform(0, 1, (N, N)) < actual_density).astype(float)
    np.fill_diagonal(A_rand, 0)
    n_active = int((A_rand > 0).sum())
    print(f'\nRandom graph seed={seed}: {n_active} edges (target density={actual_density:.3f})')
    net  = ConnectomeNetAbl(A_rand, N_CLASSES).to(DEVICE)
    hist = train_model(net, train_ld, val_ld, n_epochs=N_EPOCHS_ABL, label=f'Random{seed}')
    random_final_accs.append(hist['val_acc'][-1])
    random_histories.append(hist)

# ── Summary plot ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Learning curves
for i, (h, seed) in enumerate(zip(random_histories, range(N_RANDOM_SEEDS))):
    axes[0].plot(h['val_acc'], alpha=0.5, color='steelblue',
                 label=f'Random {seed}' if i == 0 else '_')
axes[0].plot(hist_real['val_acc'], color='tomato', linewidth=2.5, label='Real connectome')
axes[0].axhline(1/N_CLASSES, color='gray', linestyle=':', label='Chance')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('Learning Curves: Real vs. Random Graphs')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Final accuracy distribution
all_accs = random_final_accs + [real_final_acc]
labels   = [f'Rand {i}' for i in range(N_RANDOM_SEEDS)] + ['Real\nConnectome']
colors   = ['steelblue'] * N_RANDOM_SEEDS + ['tomato']
axes[1].bar(labels, all_accs, color=colors, edgecolor='black', alpha=0.8)
axes[1].axhline(np.mean(random_final_accs), color='steelblue', linestyle='--',
                label=f'Random mean={np.mean(random_final_accs):.3f}')
axes[1].set_ylabel('Final Validation Accuracy')
axes[1].set_title('Final Accuracy: Real Connectome vs. Random Sparse Graphs')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim([0, 1.1])

plt.suptitle('Exercise 5.4 — Is the Real Connectome Special?', fontsize=13)
plt.tight_layout()
plt.show()

rand_mean = np.mean(random_final_accs)
rand_std  = np.std(random_final_accs)
z_score   = (real_final_acc - rand_mean) / (rand_std + 1e-8)

print(f'\nResults:')
print(f'  Real connectome final accuracy : {real_final_acc:.3f}')
print(f'  Random graphs mean ± std        : {rand_mean:.3f} ± {rand_std:.3f}')
print(f'  Z-score of real connectome      : {z_score:.2f}')
print()
print('Interpretation:')
print('  On our small synthetic dataset the real connectome may not be dramatically')
print('  better than random graphs of the same density. This is expected because:')
print('  1. Our synthetic task was GENERATED using the connectome structure, so random')
print('     graphs of similar density perform similarly.')
print('  2. With only 8 neurons, the difference in topology between connectome and')
print('     random graphs is small (both have 5-8 edges).')
print('  3. The connectome\'s advantage emerges on tasks that specifically match its')
print('     functional organisation (which real evolution has tuned over millions of years).')

---
## Exercise 5.5 (Challenge) — C. elegans 302-Neuron ConnectomeNet

**Task:** Load the C. elegans connectome, build a ConnectomeNet, and compare parameter counts.

In [ ]:
# C. elegans connectome loading
# The full connectome is available from the OpenWorm project.
# Easiest source: a pre-parsed 302×302 numpy array from public datasets.
# Here we show the workflow; uncomment and adapt the file loading for your data.

# ── Example workflow ──────────────────────────────────────────────────────────
# Option 1: pip install connectome-tools
# from connectome_tools.connectome_reader import read_connectome
# A_cel, neuron_names = read_connectome('chemical')

# Option 2: Download Varshney et al. supplementary CSV
# import pandas as pd
# df = pd.read_csv('c_elegans_connectome.csv', index_col=0)
# A_cel = df.values.astype(float)

# ── Synthetic approximation for demonstration ─────────────────────────────────
# We approximate C. elegans statistics: 302 neurons, ~6400 synapses, density ~0.070
N_CEL     = 302
DENSITY   = 0.070
rng_cel   = np.random.default_rng(0)
A_cel_approx = (rng_cel.uniform(0, 1, (N_CEL, N_CEL)) < DENSITY).astype(float)
np.fill_diagonal(A_cel_approx, 0)
print(f'Approximate C. elegans connectome: {N_CEL} neurons, {int((A_cel_approx>0).sum())} synapses')
print(f'Actual density: {(A_cel_approx>0).sum() / (N_CEL*N_CEL):.4f}')

# ── Parameter count analysis ──────────────────────────────────────────────────
n_active_synapses = int((A_cel_approx > 0).sum())

# ConnectomeNet: two hidden layers (N_CEL → N_CEL → N_CEL → n_classes)
# Each hidden layer: n_active_synapses weights + N_CEL biases
# Head: N_CEL * n_classes + n_classes
n_classes_cel  = 4
conn_hidden    = 2 * (n_active_synapses + N_CEL)
conn_head      = N_CEL * n_classes_cel + n_classes_cel
conn_total     = conn_hidden + conn_head

# FreeNet equivalent: two fully-connected layers (N_CEL → N_CEL → N_CEL → n_classes)
free_hidden  = 2 * (N_CEL * N_CEL + N_CEL)
free_head    = N_CEL * n_classes_cel + n_classes_cel
free_total   = free_hidden + free_head

print(f'\nParameter count comparison for C. elegans connectome:')
print(f'  ConnectomeNet (sparse)  : {conn_total:,} parameters')
print(f'  FreeNet (fully-connected): {free_total:,} parameters')
print(f'  Parameter reduction     : {free_total/conn_total:.1f}× fewer in ConnectomeNet')
print(f'  Active synapse fraction : {n_active_synapses}/{N_CEL**2} = {n_active_synapses/(N_CEL**2):.4f}')

print()
print('Comparison: our 8-neuron connectome vs. C. elegans 302-neuron connectome:')
our_active = int((A > 0).sum())
our_full   = N * N
print(f'  Our connectome : {our_active:,} active / {our_full:,} total = {our_active/our_full:.4f} density')
print(f'  C. elegans     : {n_active_synapses:,} active / {N_CEL**2:,} total = {DENSITY:.4f} density')
print(f'  Scaling factor : {N_CEL}/{N} = {N_CEL/N:.0f}× more neurons')
print(f'                   {n_active_synapses}/{our_active} = {n_active_synapses/max(our_active,1):.0f}× more synapses')

# Memory estimation
bytes_conn = conn_total * 4  # float32
bytes_free = free_total * 4
print(f'\nMemory (float32):')
print(f'  ConnectomeNet: {bytes_conn/1024:.1f} KB')
print(f'  FreeNet      : {bytes_free/1024:.1f} KB')
print()
print('Key insight:')
print('  The C. elegans connectome achieves complex behaviour (chemotaxis, thermotaxis,')
print('  mating, egg-laying) with 302 neurons and ~6400 synapses — approximately')
print(f'  {free_total/conn_total:.0f}× fewer parameters than a fully-connected equivalent.')
print('  This extreme sparsity is a hallmark of biological neural circuits and suggests')
print('  that network topology carries significant inductive bias.')